# Hyperparameter Optimization

In [1]:
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_curve, confusion_matrix, roc_auc_score
from catboost import CatBoostClassifier
from skopt.space import Integer, Real

import sys 
from datetime import datetime
import shutil
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Load / Reload Selection Utility Functions

In [2]:
from utils2 import optimization as hpo

----

## Read Config File

In [3]:
config_path = Path(r'experiments')

# choose between final and development config file
config_filename = "bin_opt_dev.yml" # development
# config_filename = "bin_opt_final.yml" # final

config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict


{'experiment': {'summary': 'binary classification - hyperparameter optimization (development)',
  'classification_type': 'binary',
  'stage': 'hyperparameter_optimization',
  'tag': 'development',
  'verbosity': 1,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'name': 'catboost', 'code': 'CatBoost'},
 'param_space': {'iterations': {'min': 100, 'max': 500},
  'depth': {'min': 4, 'max': 10},
  'learning_rate': {'min': 0.01, 'max': 0.1},
  'l2_leaf_reg': {'min': 1, 'max': 9}},
 'optimization': {'scoring': 'roc_auc',
  'k_splits_outer': 3,
  'n_repeats_outer': 2,
  'k_splits_inner': 3,
  'n_iter': 5},
 'evaluation': {'confidence': 0.95},
 'final_training': {'k_splits_inner': 3, 'n_iter': 5}}

#### Set output directory

In [4]:
outputdir = config_path /  config.experiment.classification_type /  config.experiment.stage / config.model.code / config.experiment.tag 
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments\binary\hyperparameter_optimization\CatBoost\development


#### Copy config file to output directory

In [5]:
source = config_path / config_filename
destination = outputdir / config_filename
shutil.copy(source, destination)

WindowsPath('experiments/binary/hyperparameter_optimization/CatBoost/development/bin_opt_dev.yml')

## Data Loading

In [6]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
no_ncs_datacols = [c for c in data_cols if c not in D.neuro_cols]
X = dfdpn[no_ncs_datacols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 36), (190,))

## Optuna Bayes Search Optimization

In [7]:
config.param_space

namespace(iterations=namespace(min=100, max=500),
          depth=namespace(min=4, max=10),
          learning_rate=namespace(min=0.01, max=0.1),
          l2_leaf_reg=namespace(min=1, max=9))

In [8]:
hpo.model_class[config.model.name]

catboost.core.CatBoostClassifier

In [9]:
config.optimization

namespace(scoring='roc_auc',
          k_splits_outer=3,
          n_repeats_outer=2,
          k_splits_inner=3,
          n_iter=5)

In [10]:
def param_space_fn(trial):
    return  {
        "iterations": trial.suggest_int(
            "iterations", 
            config.param_space.iterations.min, 
            config.param_space.iterations.max),
        "depth": trial.suggest_int(
            "depth", 
            config.param_space.depth.min, 
            config.param_space.depth.max),
        "learning_rate": trial.suggest_float(
            "learning_rate", 
            config.param_space.learning_rate.min, 
            config.param_space.learning_rate.max, 
            log=True),
        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg", 
            config.param_space.l2_leaf_reg.min, 
            config.param_space.l2_leaf_reg.max),
    }


In [11]:
opt_results = hpo.nested_cv_youden_optuna(
    X=X.values,
    y=y.values,
    model_class=hpo.model_class[config.model.name],   # class, not an instance
    param_space_fn=param_space_fn,
    n_splits_outer=config.optimization.k_splits_outer,
    n_repeats_outer=config.optimization.n_repeats_outer,
    n_splits_inner=config.optimization.k_splits_inner,
    n_iter=config.optimization.n_iter,   # Optuna trials per outer fold
    random_state=config.experiment.random_seed,
    savedir=outputdir,
    overwrite=False,
)

optimization_results.json exists. Returning values from contents.


In [12]:
opt_results

[{'fold': 0,
  'threshold': 0.6319284707502135,
  'best_params': {'iterations': 433,
   'depth': 5,
   'learning_rate': 0.015199348301309814,
   'l2_leaf_reg': 2},
  'accuracy': 0.953125,
  'precision': 1.0,
  'sensitivity': 0.9318181818181818,
  'specificity': 1.0,
  'f1': 0.9647058823529412,
  'f2': 0.9447004608294931,
  'youden': 0.9318181818181817,
  'roc-auc': 0.9840909090909091,
  'auprc': 0.9939159032182288},
 {'fold': 1,
  'threshold': 0.5033857925077226,
  'best_params': {'iterations': 231,
   'depth': 10,
   'learning_rate': 0.04635431984752397,
   'l2_leaf_reg': 5},
  'accuracy': 0.9206349206349206,
  'precision': 0.8958333333333334,
  'sensitivity': 1.0,
  'specificity': 0.75,
  'f1': 0.945054945054945,
  'f2': 0.9772727272727273,
  'youden': 0.75,
  'roc-auc': 0.9755813953488373,
  'auprc': 0.9887960491388363},
 {'fold': 2,
  'threshold': 0.6829397944620587,
  'best_params': {'iterations': 271,
   'depth': 4,
   'learning_rate': 0.016515771639929806,
   'l2_leaf_reg': 9},


In [15]:
opt_results[0]

{'fold': 0,
 'threshold': 0.6319284707502135,
 'best_params': {'iterations': 433,
  'depth': 5,
  'learning_rate': 0.015199348301309814,
  'l2_leaf_reg': 2},
 'accuracy': 0.953125,
 'precision': 1.0,
 'sensitivity': 0.9318181818181818,
 'specificity': 1.0,
 'f1': 0.9647058823529412,
 'f2': 0.9447004608294931,
 'youden': 0.9318181818181817,
 'roc-auc': 0.9840909090909091,
 'auprc': 0.9939159032182288}

In [14]:
opt_results

[{'fold': 0,
  'threshold': 0.6319284707502135,
  'best_params': {'iterations': 433,
   'depth': 5,
   'learning_rate': 0.015199348301309814,
   'l2_leaf_reg': 2},
  'accuracy': 0.953125,
  'precision': 1.0,
  'sensitivity': 0.9318181818181818,
  'specificity': 1.0,
  'f1': 0.9647058823529412,
  'f2': 0.9447004608294931,
  'youden': 0.9318181818181817,
  'roc-auc': 0.9840909090909091,
  'auprc': 0.9939159032182288},
 {'fold': 1,
  'threshold': 0.5033857925077226,
  'best_params': {'iterations': 231,
   'depth': 10,
   'learning_rate': 0.04635431984752397,
   'l2_leaf_reg': 5},
  'accuracy': 0.9206349206349206,
  'precision': 0.8958333333333334,
  'sensitivity': 1.0,
  'specificity': 0.75,
  'f1': 0.945054945054945,
  'f2': 0.9772727272727273,
  'youden': 0.75,
  'roc-auc': 0.9755813953488373,
  'auprc': 0.9887960491388363},
 {'fold': 2,
  'threshold': 0.6829397944620587,
  'best_params': {'iterations': 271,
   'depth': 4,
   'learning_rate': 0.016515771639929806,
   'l2_leaf_reg': 9},


### Calculate Confidence Interval 

In [16]:
opt_ci_df  = hpo.mean_confidence_interval(opt_results, config, outputdir, True)
opt_ci_df

['threshold', 'accuracy', 'precision', 'sensitivity', 'specificity', 'f1', 'f2', 'youden', 'roc-auc', 'auprc']
threshold 95.0% CI: {'mean': 0.6573833064165802, 'std': 0.10396652455665704, 'ci_lower': 0.5662543041772615, 'ci_upper': 0.7485123086558988}
accuracy 95.0% CI: {'mean': 0.9158399470899471, 'std': 0.032099190807542276, 'ci_lower': 0.8877042828128677, 'ci_upper': 0.9439756113670265}
precision 95.0% CI: {'mean': 0.9580066090240509, 'std': 0.035504003165706274, 'ci_lower': 0.926886549968607, 'ci_upper': 0.9891266680794948}
sensitivity 95.0% CI: {'mean': 0.9194855532064833, 'std': 0.05066343281881791, 'ci_lower': 0.8750779123578021, 'ci_upper': 0.9638931940551645}
specificity 95.0% CI: {'mean': 0.9083333333333333, 'std': 0.08374896350934075, 'ci_lower': 0.8349254790674705, 'ci_upper': 0.9817411875991962}
f1 95.0% CI: {'mean': 0.9369496045597803, 'std': 0.025252229306958147, 'ci_lower': 0.9148154563724229, 'ci_upper': 0.9590837527471376}
f2 95.0% CI: {'mean': 0.9260539512803995, 'st

,mean,std,ci_lower,ci_upper
metric,,,,
threshold,0.657,0.104,0.566,0.749
accuracy,0.916,0.032,0.888,0.944
precision,0.958,0.036,0.927,0.989
sensitivity,0.919,0.051,0.875,0.964
specificity,0.908,0.084,0.835,0.982
f1,0.937,0.025,0.915,0.959
f2,0.926,0.039,0.892,0.960
youden,0.828,0.074,0.763,0.893
roc-auc,0.968,0.015,0.955,0.982


 -----

# Section Below are studies (not updated)

## Train final model on ALL data

Read threshold mean from file

In [ ]:
hpo_results_path = config_path /  config.experiment.classification_type / config.experiment.stage / config.model.code / config.experiment.tag / 'optimization_results.json'
with open(outputdir / "optimization_results.json", "r") as f:
    optimization_results = json.load(f)
threshold_mean = optimization_results['threshold_mean']
threshold_mean

In [ ]:
catboost_model = CatBoostClassifier(
    verbose=0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=config.experiment.random_seed, 
    thread_count=-1
)

In [ ]:
param_space = {
    'iterations': Integer(
        config.param_space.iterations.min, 
        config.param_space.iterations.max),
    'depth': Integer(
        config.param_space.depth.min, 
        config.param_space.depth.max),
    'learning_rate': Real(
        config.param_space.learning_rate.min, 
        config.param_space.learning_rate.max, 
        prior='log-uniform'),  # log-uniform better for LR
    'l2_leaf_reg': Real(
        config.param_space.l2_leaf_reg.min, 
        config.param_space.l2_leaf_reg.max, 
        prior='uniform'),
}

In [ ]:
config.final_training

In [ ]:
final_model, best_params = hpo.train_final_model(
    X=X.values, 
    y=y.values, 
    model=catboost_model,
    param_space=param_space,
    n_splits_inner=config.final_training.k_splits_inner,
    n_iter=config.final_training.n_iter, 
    random_state=config.experiment.random_seed, 
    n_jobs=1
)

In [ ]:
print(best_params) 

 -----

### Final Model Sample Prediction

In [ ]:
# 0.63 is the mean threshold, we just hardcode here
hpo.test_model(final_model, threshold_mean, X, y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
hpo.test_model(final_model, threshold_mean, X_test, y_test)

 -----

## Explainability Model Selection

Study on what model to use for counterfactuals

### Regularized Model Evaluation

In [ ]:
regularized_model = CatBoostClassifier(
    depth=4,
    l2_leaf_reg=10,
    iterations=300,
    learning_rate=0.05,
    verbose=0
)
regularized_model.fit(X,y)
hpo.test_model(regularized_model, threshold_mean, X, y)

### Unoptimized, Train-Test-Split Model Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

unoptimized_model = CatBoostClassifier(verbose=0)
unoptimized_model.fit(X_train.values, y_train.values)
hpo.test_model(unoptimized_model, threshold_mean, X_test, y_test)

### Optimized, Train-Test-Split Model Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

optsplit_model, optsplit_best_params = hpo.train_final_model(
    X=X_train.values, 
    y=y_train.values, 
    model=catboost_model,
    param_space=param_space,
    n_splits_inner=config.final_training.k_splits_inner,
    n_iter=config.final_training.n_iter, 
    random_state=config.experiment.random_seed, 
    n_jobs=1
)

In [ ]:
optsplit_model = CatBoostClassifier(**optsplit_best_params, verbose=0)
optsplit_model.fit(X_train.values, y_train.values)
hpo.test_model(unoptimized_model, threshold_mean, X_test, y_test)

*We will use this methodology for generating counterfactuals, but we will do it for 4 folds*

### Distilled Model

#### Ridge

In [ ]:
from sklearn.linear_model import Ridge

# probabilities from CatBoost
y_soft = final_model.predict_proba(X)[:,1]

distilled_model_ridge = Ridge()
distilled_model_ridge.fit(X, y_soft)

In [ ]:
hpo.test_model(distilled_model_ridge, threshold_mean, X, y, True)

In [ ]:
from sklearn.metrics import classification_report

y_pred_proba = distilled_model_ridge.predict(X)
y_pred = (y_pred_proba >= 0.63).astype(int)
print(classification_report(y, y_pred))

tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# probabilities from CatBoost
y_soft = final_model.predict_proba(X)[:,1]

distilled_model_logreg = LogisticRegression(max_iter=2000)
distilled_model_logreg.fit(X, y_soft > 0.5)

In [ ]:
hpo.test_model(distilled_model_logreg, 0.5, X, y, False)

### Surrogate Model

In [ ]:
from sklearn.linear_model import LogisticRegression

surrogate_model = LogisticRegression(
    penalty="l2",
    C=0.1,
    class_weight="balanced",
    max_iter=1000
)

surrogate_model.fit(X, y)
hpo.test_model(surrogate_model, 0.5, X, y, False)

### Mutual Info Classification

In [ ]:
hpo.plot_mutual_info(X, y)